# 04 — LLM Model Experiments

Swap the generative model and compare:
- Response quality on fixed RAG questions
- Adherence to "only answer from context" instruction
- Latency
- Prompt template sensitivity

**Models to test (all via Ollama unless noted):**
- `qwen2.5:7b` — current baseline
- `llama3.2:3b` — lighter, faster
- `mistral:7b`
- `gemma3:4b`

> Pull a model with: `ollama pull <model-name>`

In [ ]:
import sys, os, time
from pathlib import Path

repo_root = str(Path(os.getcwd()).parents[1] / "RAG_Ai_Assistant_Uni")
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import pandas as pd
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama
from app.database import search_similar_documents
from app.rag import normalize_question

## 1. Test Questions

Mix of in-scope, out-of-scope, and edge cases.

In [ ]:
TEST_QUESTIONS = [
    {"id": "q1", "question": "What services does Alphawave offer?",           "scope": "in"},
    {"id": "q2", "question": "How can I contact Alphawave?",                   "scope": "in"},
    {"id": "q3", "question": "What is the capital of France?",                 "scope": "out"},
    {"id": "q4", "question": "Does Alphawave work with healthcare companies?",  "scope": "in"},
    {"id": "q5", "question": "Who is the CEO of Alphawave?",                    "scope": "edge"},
]

print(f"{len(TEST_QUESTIONS)} test questions loaded.")

## 2. Prompt Templates

Try variations on the system instruction — stricter, more permissive, chain-of-thought.

In [ ]:
PROMPTS = {
    "baseline": PromptTemplate(
        input_variables=["context", "question"],
        template="""Answer using ONLY the Context below. If the answer isn't in the Context, say \"I don't have that information.\"

Context:
{context}

Question: {question}
Answer:"""
    ),
    "strict": PromptTemplate(
        input_variables=["context", "question"],
        template="""You are a factual assistant. You MUST only use the provided context to answer.
If the context does not contain the answer, respond with exactly: I don't have that information.
Do not add any information from your training data.

Context:
{context}

Question: {question}
Answer:"""
    ),
    "cot": PromptTemplate(
        input_variables=["context", "question"],
        template="""Use only the context below. First identify the relevant sentences, then answer.

Context:
{context}

Question: {question}
Relevant sentences: 
Answer:"""
    ),
}

print(f"Prompt variants: {list(PROMPTS.keys())}")

## 3. Model Comparison

Run each model on all test questions with the baseline prompt.
Add/remove from `MODELS` as needed.

In [ ]:
MODELS = [
    {"name": "qwen2.5:7b",  "num_ctx": 4096, "temperature": 0.1},
    # {"name": "llama3.2:3b", "num_ctx": 4096, "temperature": 0.1},
    # {"name": "mistral:7b",  "num_ctx": 4096, "temperature": 0.1},
    # {"name": "gemma3:4b",   "num_ctx": 4096, "temperature": 0.1},
]

results = []

for cfg in MODELS:
    llm = ChatOllama(model=cfg["name"], num_ctx=cfg["num_ctx"], temperature=cfg["temperature"])
    chain = PROMPTS["baseline"] | llm | StrOutputParser()
    print(f"\n{'='*60}")
    print(f"Model: {cfg['name']}")
    print(f"{'='*60}")

    for q in TEST_QUESTIONS:
        norm_q = normalize_question(q["question"])
        retrieved = search_similar_documents(norm_q, limit=5)
        context = "\n\n".join(r["content"] for r in retrieved)

        t0 = time.time()
        answer = chain.invoke({"context": context, "question": norm_q})
        latency_ms = (time.time() - t0) * 1000

        results.append({
            "model": cfg["name"],
            "q_id": q["id"],
            "scope": q["scope"],
            "question": q["question"],
            "answer": answer[:200],
            "latency_ms": round(latency_ms, 0),
        })
        print(f"  [{q['scope']:4}] {q['question'][:55]:<55} {latency_ms:6.0f}ms")
        print(f"         → {answer[:100]}")

df_results = pd.DataFrame(results)
print("\nLatency summary:")
display(df_results.groupby("model")["latency_ms"].describe())

## 4. Prompt Template Comparison

Fix the model, vary the prompt. Focus on out-of-scope questions — does the model hallucinate?

In [ ]:
FIXED_MODEL = "qwen2.5:7b"
llm = ChatOllama(model=FIXED_MODEL, num_ctx=4096, temperature=0.1)

out_of_scope = [q for q in TEST_QUESTIONS if q["scope"] == "out"]

for q in out_of_scope:
    norm_q = normalize_question(q["question"])
    retrieved = search_similar_documents(norm_q, limit=5)
    context = "\n\n".join(r["content"] for r in retrieved)
    print(f"\nQuestion: {q['question']}")
    for pname, prompt_tmpl in PROMPTS.items():
        chain = prompt_tmpl | llm | StrOutputParser()
        answer = chain.invoke({"context": context, "question": norm_q})
        print(f"  [{pname:8}] {answer[:120]}")

## 5. Temperature Sweep

Same question, sweep temperature 0.0 → 0.9. Watch for drift on factual questions.

In [ ]:
TEMP_QUESTION = "What services does Alphawave offer?"
TEMPERATURES  = [0.0, 0.1, 0.3, 0.5, 0.7, 0.9]

norm_q = normalize_question(TEMP_QUESTION)
retrieved = search_similar_documents(norm_q, limit=5)
context = "\n\n".join(r["content"] for r in retrieved)

for temp in TEMPERATURES:
    llm = ChatOllama(model=FIXED_MODEL, num_ctx=4096, temperature=temp)
    chain = PROMPTS["baseline"] | llm | StrOutputParser()
    answer = chain.invoke({"context": context, "question": norm_q})
    print(f"[temp={temp}] {answer[:150]}")
    print()